Below is a solution that:

1. reads all DJI JPGs in a selected folder

2. extracts GPS from EXIF

3. converts lat/lon to a local (x,y) in meters

4. automatically selects as many images as needed to cover the area using a spatial grid (coverage-based; not a fixed number)

5. builds a rough “pseudo-ortho” by placing images on a canvas using GPS positions (translation-only; no real orthorectification)

6. resamples the mosaic

7. tiles to 512×512

# Cell 1 — Imports & Settings

In [1]:
import rasterio
from rasterio.transform import from_origin

In [2]:
from pathlib import Path
import numpy as np
from PIL import Image
import math
import os

# ------------------------
# User settings
# ------------------------
BASE_INPUT_DIR = Path("./data_set/raw_drone_images")
BASE_OUTPUT_DIR = Path("./output/pseudo_ortho")

# Mosaic settings
MOSAIC_DOWNSCALE = 0.20    # build mosaic from smaller copies (0.15–0.30 recommended)
PIXELS_PER_METER = 3.0     # controls mosaic resolution (increase = larger mosaic)
BLEND_MODE = "average"     # "average" or "overwrite"

# Selection settings (coverage-based)
GRID_CELL_METERS = None    # if None: auto-estimate from GPS spacing
MAX_IMAGES_PER_CELL = 1    # 1 usually best (coverage selection)

# Output names
# MOSAIC_NAME = "pseudo_ortho.jpg"
# RESAMPLED_NAME = "pseudo_ortho_resampled.jpg"
# TILES_DIRNAME = "tiles_512"

# Resample + tile settings
RESAMPLE_SCALE = 1.0       # 1.0 = keep mosaic size; 0.5 = shrink by half
TILE_SIZE = 512
JPEG_QUALITY = 90

# Cell 2 — Select the input folder

In [3]:
# List valid subfolders
folders = sorted([
    p.name for p in BASE_INPUT_DIR.iterdir()
    if p.is_dir() and not p.name.startswith(".") and p.name not in ("__pycache__",)
])

if not folders:
    raise FileNotFoundError(f"No valid subfolders found in: {BASE_INPUT_DIR}")

print("Available folders:")
for i, name in enumerate(folders, 1):
    print(f"{i}. {name}")

choice = int(input(f"\nSelect folder (1-{len(folders)}): "))
SELECTED_FOLDER = folders[choice - 1]

INPUT_DIR = BASE_INPUT_DIR / SELECTED_FOLDER
OUT_DIR = BASE_OUTPUT_DIR / SELECTED_FOLDER
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"\nSelected: {SELECTED_FOLDER}")
print(f"Input:    {INPUT_DIR}")
print(f"Output:   {OUT_DIR}")

Available folders:
1. a10_segment1
2. a58_segment2



Select folder (1-2):  2



Selected: a58_segment2
Input:    data_set/raw_drone_images/a58_segment2
Output:   output/pseudo_ortho/a58_segment2


# Cell 3 — Find all JPG/JPEG

In [4]:
img_paths = sorted([
    p for p in INPUT_DIR.rglob("*")
    if (
        p.is_file()
        and p.suffix.lower() in (".jpg", ".jpeg")
        and ".ipynb_checkpoints" not in p.parts
        and "-checkpoint" not in p.name
    )
])

print(f"Found {len(img_paths)} JPG/JPEG files.")
if len(img_paths) > 0:
    print("Example:", img_paths[0].name)

Found 750 JPG/JPEG files.
Example: DJI_20260202042113_0001.JPG


# Cell 4 — Read GPS from EXIF (lat/lon)

In [5]:
def _to_float(x):
    try:
        return float(x)
    except Exception:
        return float(x[0]) / float(x[1])

def dms_to_dd(dms, ref):
    d = _to_float(dms[0])
    m = _to_float(dms[1])
    s = _to_float(dms[2])
    dd = d + m/60.0 + s/3600.0
    if ref in ("S", "W"):
        dd *= -1
    return dd

from PIL import ExifTags

def get_lat_lon(jpg_path: Path):
    """
    Robust GPS extractor for DJI JPGs.
    Returns (lat, lon) in decimal degrees or None.
    """
    try:
        im = Image.open(jpg_path)
        exif = im._getexif()
        if exif is None:
            return None

        # Map tag IDs to names
        exif_dict = {
            ExifTags.TAGS.get(k, k): v
            for k, v in exif.items()
        }

        gps_info = exif_dict.get("GPSInfo")
        if gps_info is None:
            return None

        # Map GPS sub-tags
        gps_data = {
            ExifTags.GPSTAGS.get(k, k): v
            for k, v in gps_info.items()
        }

        lat = gps_data.get("GPSLatitude")
        lat_ref = gps_data.get("GPSLatitudeRef")
        lon = gps_data.get("GPSLongitude")
        lon_ref = gps_data.get("GPSLongitudeRef")

        if not (lat and lat_ref and lon and lon_ref):
            return None

        def dms_to_dd(dms, ref):
            d = float(dms[0])
            m = float(dms[1])
            s = float(dms[2])
            dd = d + m/60 + s/3600
            if ref in ["S", "W"]:
                dd *= -1
            return dd

        return dms_to_dd(lat, lat_ref), dms_to_dd(lon, lon_ref)

    except Exception:
        return None

coords = []
valid_paths = []
missing_gps = 0

for p in img_paths:
    ll = get_lat_lon(p)
    if ll is None:
        missing_gps += 1
    else:
        valid_paths.append(p)
        coords.append(ll)

coords = np.array(coords, dtype=float)

print(f"Images with GPS: {len(valid_paths)}")
print(f"Missing GPS:     {missing_gps}")
if len(valid_paths) == 0:
    raise RuntimeError("No images with GPS found; cannot proceed with GPS-based mosaic.")

Images with GPS: 750
Missing GPS:     0


# Cell 5 — Convert lat/lon to local meters (x,y)

In [6]:
def latlon_to_local_xy_m(lat, lon, lat0, lon0):
    """
    Convert lat/lon to local meters (equirectangular approximation).
    Good for small areas (a drone site is small).
    """
    R = 6371000.0  # meters
    lat_rad = np.deg2rad(lat)
    lon_rad = np.deg2rad(lon)
    lat0_rad = math.radians(lat0)
    lon0_rad = math.radians(lon0)

    x = (lon_rad - lon0_rad) * math.cos(lat0_rad) * R
    y = (lat_rad - lat0_rad) * R
    return x, y

lat0, lon0 = coords[:,0].mean(), coords[:,1].mean()
xy = np.zeros((len(coords), 2), dtype=float)
for i, (lat, lon) in enumerate(coords):
    x, y = latlon_to_local_xy_m(lat, lon, lat0, lon0)
    xy[i] = (x, y)

print("Local XY bounds (meters):")
print("x:", xy[:,0].min(), "to", xy[:,0].max())
print("y:", xy[:,1].min(), "to", xy[:,1].max())

Local XY bounds (meters):
x: -71.02880440463173 to 38.77104710358754
y: -41.3489207123533 to 47.3877194983926


In [7]:
# print("Selected XY min/max (m):")
# print("x:", selected_xy[:,0].min(), selected_xy[:,0].max(), "range:", np.ptp(selected_xy[:,0]))
# print("y:", selected_xy[:,1].min(), selected_xy[:,1].max(), "range:", np.ptp(selected_xy[:,0]))

# print("\nFirst 5 GPS coords:")
# for p, (lat, lon) in list(zip(selected_paths, coords[selected_indices]))[:5]:
#     print(p.name, lat, lon)

In [8]:
# from PIL import Image, ExifTags
# from pathlib import Path

# test_img = img_paths[0]   # or any specific JPG

# im = Image.open(test_img)
# exif = im.getexif()

# print("Number of EXIF tags:", len(exif))

# # Print tag names
# for tag_id, value in exif.items():
#     tag_name = ExifTags.TAGS.get(tag_id, tag_id)
#     print(tag_name)

In [9]:
# import re

# def extract_xmp_block(jpg_path):
#     with open(jpg_path, 'rb') as f:
#         data = f.read()

#     start = data.find(b'<x:xmpmeta')
#     end = data.find(b'</x:xmpmeta')

#     if start == -1 or end == -1:
#         return None

#     return data[start:end+12].decode(errors="ignore")

# xmp = extract_xmp_block(test_img)

# if xmp:
#     print("XMP block found!\n")
#     print(xmp[:1000])  # print first part only
# else:
#     print("No XMP metadata found.")

In [10]:
# def extract_dji_yaw(jpg_path):
#     xmp = extract_xmp_block(jpg_path)
#     if xmp is None:
#         return None

#     flight_yaw = re.search(r'FlightYawDegree="([^"]+)"', xmp)
#     gimbal_yaw = re.search(r'GimbalYawDegree="([^"]+)"', xmp)

#     result = {}
#     if flight_yaw:
#         result["FlightYawDegree"] = float(flight_yaw.group(1))
#     if gimbal_yaw:
#         result["GimbalYawDegree"] = float(gimbal_yaw.group(1))

#     return result if result else None

# print(extract_dji_yaw(test_img))

In [11]:
# def extract_dji_pose(jpg_path):
#     xmp = extract_xmp_block(jpg_path)
#     if xmp is None:
#         return None

#     pose = {}

#     patterns = {
#         "FlightYawDegree": r'FlightYawDegree="([^"]+)"',
#         "GimbalYawDegree": r'GimbalYawDegree="([^"]+)"',
#         "GimbalPitchDegree": r'GimbalPitchDegree="([^"]+)"',
#         "GimbalRollDegree": r'GimbalRollDegree="([^"]+)"'
#     }

#     for key, pattern in patterns.items():
#         match = re.search(pattern, xmp)
#         if match:
#             pose[key] = float(match.group(1))

#     return pose if pose else None

# print(extract_dji_pose(test_img))

# Cell 6 — Auto-estimate spacing and choose images “as many as needed” for coverage

What this does:

- builds a grid over the flight area

- picks 1 photo per grid cell → “as many as needed” for coverage

- avoids exploding tile count (compared to keeping all 338)

- If we want denser coverage, either:

- reduce GRID_CELL_METERS (e.g., spacing*0.8)

- or increase MAX_IMAGES_PER_CELL to 2

In [12]:
def estimate_spacing_m(xy):
    """
    Estimate typical spacing using nearest-neighbor distance (median).
    """
    # brute force is OK for ~338; for thousands we'd use KDTree
    dmins = []
    for i in range(len(xy)):
        diffs = xy - xy[i]
        d = np.sqrt((diffs**2).sum(axis=1))
        d[i] = np.inf
        dmins.append(d.min())
    return float(np.median(dmins))

if GRID_CELL_METERS is None:
    spacing = estimate_spacing_m(xy)
    # Use something a bit larger than spacing so we don't oversample
    GRID_CELL_METERS = max(1.0, spacing * 1.2)

print(f"Estimated GPS spacing (m): ~{estimate_spacing_m(xy):.2f}")
print(f"Using grid cell size (m):  {GRID_CELL_METERS:.2f}")

# Build grid index
xmin, xmax = xy[:,0].min(), xy[:,0].max()
ymin, ymax = xy[:,1].min(), xy[:,1].max()

nx = int(np.ceil((xmax - xmin) / GRID_CELL_METERS)) + 1
ny = int(np.ceil((ymax - ymin) / GRID_CELL_METERS)) + 1

def cell_index(x, y):
    cx = int((x - xmin) // GRID_CELL_METERS)
    cy = int((y - ymin) // GRID_CELL_METERS)
    return (cx, cy)

# Assign each image to a grid cell
cell_to_indices = {}
for i, (x, y) in enumerate(xy):
    ci = cell_index(x, y)
    cell_to_indices.setdefault(ci, []).append(i)

# Select up to MAX_IMAGES_PER_CELL per cell (closest to cell center)
selected_indices = []
for (cx, cy), idxs in cell_to_indices.items():
    # cell center
    x_c = xmin + (cx + 0.5) * GRID_CELL_METERS
    y_c = ymin + (cy + 0.5) * GRID_CELL_METERS

    pts = xy[idxs]
    d = np.sqrt((pts[:,0] - x_c)**2 + (pts[:,1] - y_c)**2)
    order = np.argsort(d)

    pick = [idxs[j] for j in order[:MAX_IMAGES_PER_CELL]]
    selected_indices.extend(pick)

selected_indices = sorted(set(selected_indices))
selected_paths = [valid_paths[i] for i in selected_indices]
selected_xy = xy[selected_indices]

print(f"Selected images (coverage-based): {len(selected_paths)} out of {len(valid_paths)}")
print("Example selected:", selected_paths[0].name if selected_paths else "None")

Estimated GPS spacing (m): ~1.74
Using grid cell size (m):  2.08
Selected images (coverage-based): 617 out of 750
Example selected: DJI_20260202042113_0001.JPG


# Cell 7 — Build the pseudo-ortho mosaic by placing images on a canvas

This is a GPS-placement mosaic (translation only). It is fast and deterministic.

In [13]:
def build_rotated_gps_mosaic(selected_paths,
                             selected_xy,
                             out_path,
                             spacing,
                             downscale=0.20,
                             blend_mode="overwrite"):

    # --- auto scale ---
    im0 = Image.open(selected_paths[0]).convert("RGB")
    w0, h0 = im0.size
    tile_w = int(w0 * downscale)
    tile_h = int(h0 * downscale)

    # estimate footprint using 80% forward overlap
    footprint_m = spacing / 0.2
    pixels_per_meter = tile_w / footprint_m

    print("Auto pixels_per_meter:", pixels_per_meter)

    # --- canvas bounds ---
    x = selected_xy[:,0]
    y = selected_xy[:,1]

    xmin, xmax = x.min(), x.max()
    ymin, ymax = y.min(), y.max()

    margin_x = (tile_w / pixels_per_meter)
    margin_y = (tile_h / pixels_per_meter)

    xmin -= margin_x
    xmax += margin_x
    ymin -= margin_y
    ymax += margin_y

    canvas_w = int((xmax - xmin) * pixels_per_meter)
    canvas_h = int((ymax - ymin) * pixels_per_meter)

    print("Canvas size:", canvas_w, "x", canvas_h)

    canvas = np.zeros((canvas_h, canvas_w, 3), dtype=np.uint8)

    def to_px(xm, ym):
        px = int((xm - xmin) * pixels_per_meter)
        py = int((ymax - ym) * pixels_per_meter)
        return px, py

    for i, (p, (xm, ym)) in enumerate(zip(selected_paths, selected_xy), 1):

        pose = extract_dji_pose(p)
        yaw = pose["GimbalYawDegree"] if pose and "GimbalYawDegree" in pose else 0.0

        im = Image.open(p).convert("RGB")
        im = im.resize((tile_w, tile_h), Image.Resampling.BILINEAR)

        # rotate image
        im = im.rotate(-yaw, expand=True, resample=Image.Resampling.BICUBIC)

        arr = np.array(im)

        cx, cy = to_px(xm, ym)

        h_img, w_img = arr.shape[:2]

        x1 = cx - w_img // 2
        y1 = cy - h_img // 2
        x2 = x1 + w_img
        y2 = y1 + h_img

        # clip
        ix1 = max(0, x1)
        iy1 = max(0, y1)
        ix2 = min(canvas_w, x2)
        iy2 = min(canvas_h, y2)

        if ix2 <= ix1 or iy2 <= iy1:
            continue

        sx1 = ix1 - x1
        sy1 = iy1 - y1
        sx2 = sx1 + (ix2 - ix1)
        sy2 = sy1 + (iy2 - iy1)

        patch = arr[sy1:sy2, sx1:sx2]

        # canvas[iy1:iy2, ix1:ix2] = patch
        target = canvas[iy1:iy2, ix1:ix2]

        # Create mask: keep only non-black pixels
        mask = np.any(patch > 5, axis=2)  # threshold avoids very dark noise
        
        target[mask] = patch[mask]

        alpha = 0.7
        target[mask] = (
            alpha * patch[mask] +
            (1 - alpha) * target[mask]
        ).astype(np.uint8)

        if i % 20 == 0 or i == len(selected_paths):
            print(f"Placed {i}/{len(selected_paths)}")

    # meters per pixel
    pixel_size = 1.0 / pixels_per_meter
    
    # affine transform (top-left origin)
    transform = from_origin(
        xmin,   # west (top-left x)
        ymax,   # north (top-left y)
        pixel_size,
        pixel_size
    )
    
    with rasterio.open(
        out_path.with_suffix(".tif"),
        "w",
        driver="GTiff",
        height=canvas.shape[0],
        width=canvas.shape[1],
        count=3,
        dtype=canvas.dtype,
        crs="EPSG:32610",
        transform=transform,
    ) as dst:
        for band in range(3):
            dst.write(canvas[:, :, band], band + 1)
    
    print("Saved GeoTIFF:", out_path.with_suffix(".tif"))
    print("Saved:", out_path)

In [14]:
import re
from PIL import Image

def extract_xmp_block(jpg_path):
    with open(jpg_path, "rb") as f:
        data = f.read()
    start = data.find(b"<x:xmpmeta")
    end = data.find(b"</x:xmpmeta")
    if start == -1 or end == -1:
        return None
    return data[start:end+12].decode(errors="ignore")

def extract_dji_pose(jpg_path):
    """
    Returns dict with DJI pose fields if present in XMP.
    Example keys: FlightYawDegree, GimbalYawDegree, GimbalPitchDegree, GimbalRollDegree
    """
    xmp = extract_xmp_block(jpg_path)
    if xmp is None:
        return None

    patterns = {
        "FlightYawDegree": r'FlightYawDegree="([^"]+)"',
        "GimbalYawDegree": r'GimbalYawDegree="([^"]+)"',
        "GimbalPitchDegree": r'GimbalPitchDegree="([^"]+)"',
        "GimbalRollDegree": r'GimbalRollDegree="([^"]+)"',
    }

    pose = {}
    for key, pat in patterns.items():
        m = re.search(pat, xmp)
        if m:
            pose[key] = float(m.group(1))

    return pose if pose else None

# quick test
print("Pose example:", extract_dji_pose(img_paths[0]))

Pose example: {'FlightYawDegree': -100.9, 'GimbalYawDegree': -100.5, 'GimbalPitchDegree': -89.9, 'GimbalRollDegree': 0.0}


In [15]:
mosaic_path = OUT_DIR / "pseudo_ortho_rotated.png"

build_rotated_gps_mosaic(
    selected_paths,
    selected_xy,
    mosaic_path,
    spacing,
    downscale=0.20
)

Auto pixels_per_meter: 188.55395036707984
Canvas size: 23852 x 18896
Placed 20/617
Placed 40/617
Placed 60/617
Placed 80/617
Placed 100/617
Placed 120/617
Placed 140/617
Placed 160/617
Placed 180/617
Placed 200/617
Placed 220/617
Placed 240/617
Placed 260/617
Placed 280/617
Placed 300/617
Placed 320/617
Placed 340/617
Placed 360/617
Placed 380/617
Placed 400/617
Placed 420/617
Placed 440/617
Placed 460/617
Placed 480/617
Placed 500/617
Placed 520/617
Placed 540/617
Placed 560/617
Placed 580/617
Placed 600/617
Placed 617/617
Saved GeoTIFF: output/pseudo_ortho/a58_segment2/pseudo_ortho_rotated.tif
Saved: output/pseudo_ortho/a58_segment2/pseudo_ortho_rotated.png
